# Day 68: NLP with Deep Learning
## Word Embeddings, Attention & the Transformer Revolution

---

# PART 1: THEORY

## 1. From Words to Vectors

Machines need numbers. How do you represent words mathematically?

**Bag of Words / TF-IDF:** Count word frequencies. Each word is a unique index.
- Problem: "cat" and "kitten" are as different as "cat" and "rocket" — no notion of similarity.

**Word Embeddings (Word2Vec, 2013):** Each word -> dense vector (100-300D). Similar words have similar vectors.
- **Famous property:** King - Man + Woman = Queen!
- The vector arithmetic captures semantic relationships

**How Word2Vec learns:** Predict a word from its context (CBOW) or predict context from a word (Skip-gram). After training on billions of words, similar words cluster together.

## 2. The Problem with RNNs for NLP

RNNs/LSTMs process words ONE AT A TIME, sequentially. This means:
- **Slow:** Can't parallelize (each step depends on the previous)
- **Long-range dependencies still hard:** Even LSTM struggles with very long text
- **Information bottleneck:** All information must pass through a single hidden state

## 3. The Transformer (2017) — "Attention Is All You Need"

The paper that changed everything. Key innovations:

**Self-Attention:** Each word "attends" to ALL other words simultaneously.
- "The cat sat on the mat because it was tired"
- Self-attention learns that "it" refers to "cat" (not "mat")

**Multi-Head Attention:** Multiple attention "heads" learn different relationships:
- Head 1: Subject-verb relationships
- Head 2: Pronoun resolution
- Head 3: Adjective-noun pairs

**Positional Encoding:** Since there's no sequential processing, add position information to the input.

## 4. BERT, GPT, and Modern NLP

- **BERT (2018):** Bidirectional — understands context from both left and right. Best for: classification, NER, Q&A
- **GPT (2018+):** Autoregressive — predicts next word. Best for: text generation, chatbots, writing
- **T5, LLaMA, Claude, Gemini:** Even larger and more capable

**We'll use HuggingFace** — the library that makes thousands of pre-trained models accessible in 3 lines of code.

---

# PART 2: PRACTICAL

## 5. Word Embeddings in Keras

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load IMDB
vocab_size = 10000; max_len = 200
(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=vocab_size)
X_train_p = keras.preprocessing.sequence.pad_sequences(X_train, maxlen=max_len)
X_test_p = keras.preprocessing.sequence.pad_sequences(X_test, maxlen=max_len)

# The Embedding layer learns word vectors during training!
embedding_model = keras.Sequential([
    layers.Embedding(vocab_size, 128, input_length=max_len),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])
embedding_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
embedding_model.summary()

print(f"\nThe Embedding layer learns a 128-dim vector for each of the {vocab_size} words")
print(f"That's {vocab_size * 128:,} numbers that represent word meanings!")


In [ ]:
# Train and show that embeddings learned something
embedding_model.fit(X_train_p, y_train, epochs=5, batch_size=64, validation_split=0.2, verbose=1)
_, acc = embedding_model.evaluate(X_test_p, y_test, verbose=0)
print(f"\nTest Accuracy: {acc:.3f}")


## 6. Pre-trained GloVe Embeddings

In [ ]:
# Load pre-trained GloVe embeddings (they auto-download!)
# GloVe was trained on billions of words — much better than training from scratch

# For demonstration: use a small pre-trained embedding
# In practice, download glove.6B.zip and use the 100d vectors

# Build model with pre-trained embeddings (concept demo)
embedding_dim = 100

# Normally you would load GloVe vectors here and create an embedding matrix
# embedding_matrix = load_glove_vectors('glove.6B.100d.txt', word_index, embedding_dim)

# Then pass to Embedding layer:
# layers.Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=False)

print("In practice, GloVe embeddings are loaded from a file and set as Embedding layer weights")
print("This gives a huge boost for small datasets!")
print("\nDownload from: https://nlp.stanford.edu/projects/glove/")


## 7. HuggingFace — Instant NLP with Pre-trained Models

In [ ]:
# Install and use HuggingFace transformers
try:
    from transformers import pipeline
    print("Transformers ready!")
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'transformers', '-q'])
    from transformers import pipeline
    print("Transformers installed!")

# Sentiment Analysis — no training needed!
sentiment = pipeline('sentiment-analysis')

texts = [
    "I absolutely loved this course! Best learning experience ever.",
    "This was a complete waste of time and money. Terrible.",
    "The food was okay but the service could have been better.",
]

print("Zero-shot sentiment analysis (no training!):")
for text in texts:
    result = sentiment(text)[0]
    emoji = ":-)" if result['label'] == 'POSITIVE' else ":-("
    print(f"  {emoji} ({result['score']:.2%}): '{text[:60]}...'")


In [ ]:
# Named Entity Recognition
ner = pipeline('ner', grouped_entities=True)
text = "Hardik studied at IIT Delhi and works at Google in Bangalore."
print(f"Text: '{text}'")
print("Entities found:")
for entity in ner(text):
    print(f"  - {entity['word']} -> {entity['entity_group']} ({entity['score']:.2%})")


In [ ]:
# Text Generation with GPT-2
try:
    generator = pipeline('text-generation', model='gpt2')
    prompt = "Machine learning is changing the world because"
    result = generator(prompt, max_length=50, num_return_sequences=1)[0]
    print(f"Prompt: '{prompt}'")
    print(f"Generated: '{result['generated_text']}'")
except Exception as e:
    print(f"GPT-2 requires more setup. Error: {e}")
    print("For now, use smaller models or cloud APIs.")


---

# PART 3: EXERCISES

In [ ]:
# Exercise 1: Zero-shot classification — classify ANY text
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

# Describe what you want to classify — no training!
texts = [
    "The new iPhone has an amazing camera and battery life",
    "The stock market crashed today due to inflation fears",
    "The cricket match was thrilling with a last-ball six",
]

labels = ['technology', 'finance', 'sports', 'entertainment', 'health']

for text in texts:
    result = classifier(text, labels)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    print(f"\nText: '{text[:60]}...'")
    print(f"Classified as: {top_label} ({top_score:.1%})")


In [ ]:
# Exercise 2: Summarize text with a pre-trained model
summarizer = pipeline('summarization', model='sshleifer/distilbart-cnn-12-6')

long_text = """
Deep learning is part of a broader family of machine learning methods based on
artificial neural networks with representation learning. Learning can be supervised,
semi-supervised or unsupervised. Deep-learning architectures such as deep neural
networks, deep belief networks, deep reinforcement learning, recurrent neural
networks, convolutional neural networks and transformers have been applied to
fields including computer vision, speech recognition, natural language processing,
machine translation, bioinformatics, drug design, medical image analysis, climate
science, material inspection and board game programs, where they have produced
results comparable to and in some cases surpassing human expert performance.
"""

try:
    summary = summarizer(long_text, max_length=50, min_length=20)
    print(f"Original length: {len(long_text)} chars")
    print(f"Summary: {summary[0]['summary_text']}")
except Exception as e:
    print(f"Summarization model needs downloading. Try running again.")


## Key Takeaways

- **Word embeddings** map words to dense vectors where similar words are close
- **Transformer** replaced RNNs with self-attention — processes all words simultaneously
- **BERT** understands text (classification, Q&A, NER)
- **GPT** generates text (chatbots, writing, code)
- **HuggingFace** gives you thousands of pre-trained models for free
- **Zero-shot** = classify without ANY training data — just describe what you want
- Modern NLP: 90% of the time, just use a pre-trained model

**Tomorrow:** Deep Learning Mini Project — Cats vs Dogs end-to-end!